In [1]:
import requests
import json
import re
import pandas as pd

In [2]:
def get_uniprot(accession):
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
    http_function = requests.get
    return http_function(endpoint)

In [3]:
def uniprot_parse_response(resp):
    if resp.status_code != 200:
        return {"error": f"HTTP error: {resp.status_code}"}
    resp = resp.json()
    output = {}
    acc = resp['primaryAccession']
    organism = resp['organism']['scientificName']
    geneInfo = resp['genes']
    sequenceInfo = resp['sequence']
    output[acc] = {'organism':organism, 'geneInfo':geneInfo, 'sequenceInfo':sequenceInfo, 'type':'protein'}
    return output

In [4]:
def get_ensembl(id):
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}?expand=1"
    http_function = requests.get
    return http_function(endpoint, headers={ "Content-Type" : "application/json"})


def ensembl_parse_response(resp):
    if resp.status_code != 200:
        return {"error": f"HTTP error: {resp.status_code}"}
    resp = resp.json()
    output = {}
    acc = resp['id']
    output[acc] = {
        'object_type' : resp['object_type'],
        'assembly_name' : resp['assembly_name'],
        'species' : resp['species'],
        'db_type' : resp['db_type'],
        'biotype' : resp['biotype'],
        'display_name' : resp['display_name'],
        'id' : acc,
        'description' : resp['description'],
        'canonical_transcript' : resp['canonical_transcript'],
        'source' : resp['source']}
    return output

In [21]:
def identify_database(id_string):
    uniprot_patterns = [
        r'^[A-NR-Z][0-9][A-Z0-9]{3}[0-9]$',
        r'^[OPQ][0-9][A-Z0-9]{3}[0-9]$',
        r'^[A-Z0-9]{6,10}$']
    ensembl_patterns = [
        r'^ENS[A-Z]*[G|T|P|E][0-9]{11}$',
        r'^ENS[A-Z]*[0-9]{11}$',
        r'^[A-Z]{3,4}[0-9]{8,11}$']
    for pattern in uniprot_patterns:
        if re.match(pattern, id_string):
            return 'uniprot'
    for pattern in ensembl_patterns:
        if re.match(pattern, id_string):
            return 'ensembl'
    return 'unknown'

def main(ids: list):
    output = {}
    for id_string in ids:
        db_type = identify_database(id_string)
        if db_type == 'uniprot':
            response = get_uniprot(id_string)
            parsed_info = uniprot_parse_response(response)
            output.update(parsed_info)
        elif db_type == 'ensembl':
            response = get_ensembl(id_string)
            parsed_info = ensembl_parse_response(response)
            output.update(parsed_info)    
        else:
            output[id_string] = {"error": f"Unknown database type for ID: {id_string}"}
    return pd.DataFrame.from_dict(output, orient='index')

In [6]:
get_uniprot('P11473')

<Response [200]>

In [7]:
get_uniprot('helloworld')

<Response [400]>

In [8]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [9]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [10]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [11]:
get_ensembl('helloworld')

<Response [400]>

In [12]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [13]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

In [22]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,Unknown database type for ID: hello,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
